# Workflow

To process a document, we intend to the following things:
1. Ingest the data from the file.
2. Break the data into chunks that easily fit into the context window of the model.
3. Create vector embedding of the chunks for the vector database to retrieve it.
4. Put the chunks, metadata and embeddings to a vector DB.
5. Retrieve the chunks by querying the database.
6. Feed the retrieved data to an LLM to compile these chunks and give us a summary.

# Data Ingestion

In [1]:
import os                                                               # For directory and file creation
import re                                                               # For data cleaning using regular expressions

In [2]:
# Libraries that are required for document loading
from langchain_core.documents import Document
from langchain_community.document_loaders import TextLoader             # For loading tect
from langchain_community.document_loaders import DirectoryLoader        # For loading directories
from langchain_community.document_loaders import PyMuPDFLoader          # For loading PDFs
from langchain_community.document_loaders import PyPDFLoader            # For loading PDFs
from langchain_text_splitters import RecursiveCharacterTextSplitter     # For chunking
from langchain_ollama.chat_models import ChatOllama                     # As we are calling local LLM, we require to import this

/home/rahu_g/.anaconda3/envs/LLM/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Libraries required for vector store
import numpy as np                                                      # For storing embeddings as ndarray
import sklearn                                                          # Required by sentence_transformers
import chromadb                                                         # The vector store
import uuid                                                             # For generating unique document IDs
from typing import List, Dict, Any, Tuple                               # For data storing and manipulation. These are non-primitive data types (object of List, Dict, Any and Tuple). Required as loaders return non-primitive datatypes.
from sklearn.metrics.pairwise import cosine_similarity                  # For calculating similarity between query and stored document embeddings
from sentence_transformers import SentenceTransformer                   # Embeddings will be generated from transformers provided by this library

E0000 00:00:1776515294.444812 3516773 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1776515294.444837 3516773 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_rejected' registered more than once. Ignoring later registration.
E0000 00:00:1776515294.444839 3516773 instrument.cc:563] Metric with name 'grpc.resource_quota.connections_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1776515294.444840 3516773 instrument.cc:563] Metric with name 'grpc.resource_quota.instantaneous_memory_pressure' registered more than once. Ignoring later registration.
E0000 00:00:1776515294.444841 3516773 instrument.cc:563] Metric with name 'grpc.resource_quota.memory_pressure_control_value' registered more than once. Ignoring later registration.


## 1. Data Injestion

Loading all the PDF files from "./data/pdf_files"

In [4]:
dir_loader = DirectoryLoader(   "./data/pdf_files", 
                                glob="**/*.pdf",
                                loader_cls=PyPDFLoader,
                                show_progress=True
                            )
pdf_documents = dir_loader.load()

100%|██████████| 5/5 [00:02<00:00,  2.25it/s]


## 2. Embedding

RAG queries documents by their embeddings in a fuzzy way instead of some form of a discrete key.

Embeddings are basically the weights of the second last layer of a text classifier. 
It contains almost all information of the text fed to the text classifier, but in a fixed size and format.
As embeddings are essentially lists of floating point numbers, they can be used to classify and distinguish one sentence from other.

Embeddings can also be conceptualised as n-dimentional representation of the sentence, where each dimention may vaguely represent a concept. 
Two sentences can be similar if they have close values at same indices, representing they are conceptually alike.

A query in RAG is also passed throught this truncated text classifier, which generates it's embeddings. 
The embedding of the query is then matched with the embeddings of all the stored documents.
The document or chunk which has the highest similarity with respect to the embedding is considered to be most closely related to the query.


Here we define the Embedding_Manager class which is responsible for generation of the embeddings.

In [5]:
class Embedding_Manager:
    # To handle document embedding generation
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):

        # Initialize embedding manager
        # model_name: Huggingface model name for sentence embedding
        
        self.model_name = model_name
        self.model = None
        self._load_model()
    
    def _load_model(self):
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimention {self.get_embedding_dimension()}")
        except Exception as e:
            print(f"Failed loading model {self.model_name}: {e}")
            raise
    
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        
        # Generate embeddings for a list of texts
        # texts: List of text strings to embed
        # returns: numpy array of embeddings with shape (len(texts), embedding_dim)
        
        if not self.model:
            raise ValueError("Model not loaded")
        print(f"Generating embeddings for {len(texts)} texts.")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    
    def get_embedding_dimension(self) -> int:
        """
            Get the embedding dimension of the model
        """
        if not self.model:
            raise ValueError("Model not loaded")
        return self.model.get_sentence_embedding_dimension()

In [6]:
embedding_manager = Embedding_Manager()

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14359.28it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimention 384


## 3. Chunking

LLMs have a limited context window. 
Any sentence exceeding the limit of the context window will be rejected from processing by the LLM. 
So the text is divided into chunks of managable size, ideally just a bit less than the context window limit.

The splitting is done on a fixed size which is a bit less than the context window of the LLM. An overlap is also considered which incorporates last characters of the previous chunk body (not considering it's overlap) and first characters of next chunk body in order to preserve context.

There are various text splitters provided by LangChain library which can be employed for chunking.<br>
These text splitters fall under the following categories:<br>
    1. **Core text splitters**: These text splitters split the text by considering only characters determined by the programmer.<br>
        &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;a. CharacterTextSplitter: splits text by using specified delimiters. The used can set them to be '\n', '\n\n', '\t ', ' ', etc., or any other character.<br>
        &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;b. RecursiveCharacterTextSplitter: Recursively tries to split using progressively “weaker” separators. It first splits paragraphs ('\n\n'), then it splits lines ('\n'), then spaces (" ") and then special characters ("").<br>
    2. **Token based splitters**: These chunks made are alligned automatically to the context window of the LLM.<br>
        &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;a. TokenTextSplitter: Splits strictly by token count. It uses tokenizers like TikToken.<br>
        &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;b. SentenceTransformersTokenTextSplitter: Optimized for sentence-transformer models.<br>
        &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;c. SpacyTextSplitter: Uses spaCy sentence boundaries. It is more linguistic awareness than raw tokens.<br>
    3. **Structure aware splitters**: These respect document structure instead of blind chunking.<br>
        &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;a. MarkdownHeaderTextSplitter: Splits by markdown headers (#, ##, etc.)<br>
        &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;b. HTMLHeaderTextSplitter: It splits text based on explicit header tags.<br>
        &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;c. HTMLSectionSplitter: It splits HTML into logical sections using DOM structure, not just headers.<br>
        &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;d. RecursiveJsonSplitter: Respect the JSON tree first. Only break nodes when they grow too large. Splitting happens by recursively descending the tree and partitioning it into smaller JSON chunks that fit within a size constraint.<br>
    4. **Code specific splitters**: Designed for programming languages.<br>
        &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;a. LanguageTextSplitter: General-purpose code splitter, supports multiple coding languages.<br>
        &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;b. PythonCodeTextSplitter: Python specific code splitter.<br>
        &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;c. JSCodeTextSplitter: JavaScript specific code splitter.<br>
        &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;d. MarkdownCodeTextSplitter: Markdown specific code splitter.<br>
    5. **Document specific splitters**: Splitters based on source formatting.<br>
        &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;a. LatexTextSplitter: Splits based on LaTeX sections.<br>
        &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;b. RSTTextSplitter: For reStructuredText.<br>
        &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;c. PythonAstTextSplitter: Uses AST parsing instead of raw text.<br>
    6. **Semantic splitters**: Tries to preserve meaning, not just size.<br>
        &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;a. SemanticChunker: Uses embeddings similarity. Groups semantically related sentences. Requires embedding model.<br>
    7. **Specialised splitters**:<br>
        &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;a. NLTKTextSplitter: Uses Natural Language Toolkit. Sentence-aware splitting.<br>
        &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;b. SentenceSplitter: Basic sentence boundary splitter.<br>
    8. **Experimental splitters**: Splitters which are deemed experimental or custom or user-defined splitters constructed by via inheriting **TextSplitter**.<br>


In [7]:
def recursive_split_documents(documents,chunk_size=2000,chunk_overlap=200):
    # Split documents into smaller chunks for better RAG performance.
    
    # Parameters:
    # - chunk_size: Maximum characters per chunk (adjust based on your LLM)
    # - chunk_overlap: Characters to overlap between chunks (preserves context)
    
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,                              # Each chunk: ~1000 characters
        chunk_overlap=chunk_overlap,                        # 200 chars overlap for context
        length_function=len,                                # How to measure length
        separators=["\n", " ", ""]                          # Split hierarchy
    )
    # Actually split the documents
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show what a chunk looks like
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [8]:
chunks = recursive_split_documents(pdf_documents,chunk_size=1000 ,chunk_overlap=200)

Split 58 documents into 362 chunks

Example chunk:
Content: Merging Feed-Forward Sublayers for Compressed Transformers
Neha Verma1 Kenton Murray1,2 Kevin Duh1,2
1Center for Language and Speech Processing
2Human Language Technology Center of Excellence
Johns Ho...
Metadata: {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-04-01T00:13:02+00:00', 'author': '', 'keywords': '', 'moddate': '2025-04-01T00:13:02+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'data/pdf_files/paper1.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1'}


Some artifacts were present post chunking, to remove it, we perform regular expression based replacement.

In [9]:
def clean_chunk_text(text: str) -> str:
    # 1. Fix hyphenated line breaks (critical for PDFs)
    text = re.sub(r'-\n', '', text)
    
    # 2. Replace single newlines with space
    text = re.sub(r'(?<!\n)\n(?!\n)', ' ', text)
    
    # 3. Normalize multiple newlines (optional)
    text = re.sub(r'\n+', '\n\n', text)
    
    # 4. Normalize whitespace
    text = re.sub(r'[ \t]+', ' ', text)
    
    return text.strip()

cleaned_chunks = []

for doc in chunks:
    cleaned_text = clean_chunk_text(doc.page_content)
    
    cleaned_doc = doc.__class__(
        page_content=cleaned_text,
        metadata=doc.metadata
    )
    
    cleaned_chunks.append(cleaned_doc)

chunks = cleaned_chunks

## 6. Vector Store

We are using ChromaDB to store the chunks and the respective embeddings.

In [10]:
class VectorStore:
    # Manages document embeddings in a ChromaDB vector store

    def __init__(self, collection_name: str = "chunks", persist_directory: str = "./data/vector_store"):
        # Initialize the vector store
        # collection_name: Name of the ChromaDB collection
        # persist_directory: Directory to persist the vector store

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
    
    def _initialize_store(self):
        # Initialize ChromaDB client and collection
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path = self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection  (   name=self.collection_name, 
                                                                        metadata={"description":"PDF document embeddings for RAG"}
                                                                    )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
    
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        # Add documents and their embeddings to vector store
        # documents: List of LangChain documents
        # embeddings: Corresponding embeddings for the documents
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents not equal to number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store.")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc,embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())
            
            # Metadata and content can be used for filtering and retrieval later
            metadatas.append(metadata)
        
        # Add to collection
        try:
            self.collection.add(
                ids = ids,
                embeddings = embeddings_list,
                metadatas = metadatas,
                documents = documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to the vector store: {e}")
            raise

In [11]:
vector_store = VectorStore()
print(f"Type of vector_store: {type(vector_store)}")

Vector store initialized. Collection: chunks
Existing documents in collection: 3620
Type of vector_store: <class '__main__.VectorStore'>


## 7. Extracting all texts from the chunks and creating embeddings

Convert the text to embeddings

In [12]:
texts = [doc.page_content for doc in chunks]
len(texts)

362

Generate embeddings for the document chunks


In [13]:
embeddings = embedding_manager.generate_embeddings(texts)
len(embeddings)

Generating embeddings for 362 texts.


Batches: 100%|██████████| 12/12 [00:00<00:00, 18.21it/s]

Generated embeddings with shape: (362, 384)


362

Store in vector store

In [14]:
vector_store.add_documents(chunks,embeddings)

Adding 362 documents to vector store.
Successfully added 362 documents to vector store
Total documents in collection: 3982


## 8.Retrieval from Vector Store

In [15]:
class RAGRetriever:
    #Handle query-based retrieval from vector store
    def __init__(self, vector_store: VectorStore, embedding_manager: Embedding_Manager):
        # Initialize the RAG retriever
        # vector_store: Vector store instance
        # embedding_manager: Embedding manager instance
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Tuple[str, Dict[str, Any]]]:
        
        # Retrieve relevant documents for a query
        # query: User query string
        # top_k: Number of top results to return
        # score_threshold: Minimum similarity score for retrieval
        # returns: List of tuples (document content, metadata)
        
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate embedding for the query. The vector database searches via embeddings, so the embeddings of the query is required.
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Search in vector store
        try:
            
            # Query the database with the query and retrieve the top 'k' documents.

            # The vector store replies back with the "document content", "metadata", and "distance from query" of all the documents stored in the vector store.
            # This returned list is sorted from the least distanced document to the most.
            # if the number of results 'n' is mentioned in the function call, only the top 'n' results will be retrieved.
            
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:            # In the retrieved content, if document exists and it has some content, get the document.
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                # For each document/chunk which returned by the vector store, find the similarity score. 
                # If similarity score is within range, the document is to be considered.
                # We store the ID, content, metadata, similarity score, distance and rank of the document retrieved.

                # The the details and content of the vector retrieved is then to be stored onto a list of tuples. 
                # Each tuple will hold a dictionary, which will contain the fields like 'id', 'content', 'metadata', 'similarity', 'distance', 'rank' and their associated content.
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance, so similarity = 1 - distance)
                    similarity_score = 1 - distance
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id' : doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                print(f"Retrieved {len(retrieved_docs)} documents above the similarity threshold.")
            else:
                print(f"No documents retrieved for the query.")
                
            return retrieved_docs
        
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

In [16]:
rag_retriever = RAGRetriever(vector_store, embedding_manager)
print(f"Type of rag_retriever: {type(rag_retriever)}")

Type of rag_retriever: <class '__main__.RAGRetriever'>


In [17]:
rag_retriever.retrieve("Event Prediction", top_k=3, score_threshold=0.01)

Retrieving documents for query: 'Event Prediction'
Top K: 3, Score threshold: 0.01
Generating embeddings for 1 texts.


Batches: 100%|██████████| 1/1 [00:00<00:00, 17.80it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents above the similarity threshold.


[{'id': 'doc_b1bc69e0_143',
  'content': 'pared to standard all-pair attention. 3) Extensive experiments on two actual EHR datasets demonstrate that our proposed model outperforms several benchmarks. Methodology Problem definition Consider a temporal sequence Q of events denoted as ⟨e1, ..., ei, ..., eL⟩, where L represents the length of the sequence. Each event, ei, can be characterized by a pair (ti, ki): ti signifies the event time, and ki ∈ {1, 2, ..., K} indicates the event type, with K denoting the total number of type classes. The objective of the event prediction problem is to predict the subsequent eventeL+1 = (tL+1, kL+1). It is important to note that the time of each event,ti, is irregular, which means that events do not occur at fixed intervals. These event times can exhibit patterns across various temporal scales. For instance, clinical operational events like medication administration may be recorded at minute intervals within an operation room but may be recorded every f

In [18]:
rag_retriever.retrieve("what is feature based cycle aware time positional encoding", top_k=3, score_threshold=0.01)

Retrieving documents for query: 'what is feature based cycle aware time positional encoding'
Top K: 3, Score threshold: 0.01
Generating embeddings for 1 texts.


Batches: 100%|██████████| 1/1 [00:00<00:00, 235.86it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents above the similarity threshold.


[{'id': 'doc_8a84ea08_147',
  'content': 'to capture the relative temporal order of events in TPPs. Existing methods can be classified into fixed (Vaswani et al. 2017) and learned encoding (Kazemi et al. 2019; Xu et al. 2020; Zhang et al. 2020; Xu et al. 2019; Li et al. 2021; Dikeoulias, Amin, and Neumann 2022; Shaw, Uszkoreit, and Vaswani 2018; Raffel et al. 2020), but they fail to learn event cycles based on event features. Research highlights the importance of incorporating semantic features to accurately represent periodic patterns in real-world phenomena (Ke, He, and Liu 2021; Zhang, Lee, and Lee 2019). To effectively capture complex cyclic patterns in irregular time sequences, we introduce a novel Feature-based Cycle-aware Time Positional Encoding (FCPE), which integrates these essential semantic aspects into the encoding of time intervals between events. Formally, time positional encoding can be described as a function P : T → Rd×1, mapping the time domain T ⊂ R',
  'metadata': 

## 9. Retrieval Augmented Generation

We choose the model which will perform the summary generation. This is a local Ollama based model.

In [19]:
model = ChatOllama(model="gemma4:e4b")

Here we define a simple RAG function which will retrieve the context similar to the given query and generate a summary as a response.

In [24]:
def rag_v1(query, retriever, llm, top_k = 3):
    print(type(llm))
    results = retriever.retrieve(query,top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return f"No relevant context found for the query: {query}"
    prompt = f"Use the following context to answer the question:\n\n{context}\n\nQuestion: {query}\nAnswer:"
    response = llm.invoke(prompt.format(context=context, question = query))
    return response

In [25]:
def rag_v2(query, retriever, llm, top_k = 3):
    print(type(llm))
    results = retriever.retrieve(query,top_k=top_k)
    context = "\n\n".join([f"Document {i+1}:\nContent: {res['content']}\nMetadata: {res['metadata']}\nSimilarity: {res['similarity']:.4f}" for i, res in enumerate(results)])
    prompt = f"Use the following retrieved documents to answer the question:\n\n{context}\n\nQuestion: {query}\nAnswer:"
    response = llm.invoke(prompt)
    return response

In [26]:
print(rag_v1("what is feature based cycle aware time positional encoding", rag_retriever, model, top_k=3).content)

<class 'langchain_ollama.chat_models.ChatOllama'>
Retrieving documents for query: 'what is feature based cycle aware time positional encoding'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts.


Batches: 100%|██████████| 1/1 [00:00<00:00, 79.07it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents above the similarity threshold.


Based on the context, the Feature-based Cycle-aware Time Positional Encoding (FCPE) is:

A **novel** encoding introduced to effectively capture complex cyclic patterns in irregular time sequences. It works by **integrating essential semantic aspects** (semantic features) into the encoding of the time intervals between events.

This method addresses a limitation of existing time positional encoding techniques, which the text notes fail to learn event cycles based on event features.


In [27]:
print(rag_v2("what is feature based cycle aware time positional encoding", rag_retriever, model, top_k=3).content)

<class 'langchain_ollama.chat_models.ChatOllama'>
Retrieving documents for query: 'what is feature based cycle aware time positional encoding'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts.


Batches: 100%|██████████| 1/1 [00:00<00:00, 65.52it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents above the similarity threshold.


The Feature-based Cycle-aware Time Positional Encoding (FCPE) is a novel method introduced to effectively capture complex cyclic patterns within irregular time sequences.

According to the documents, FCPE functions by:
*   Integrating essential **semantic aspects** (semantic features) into the encoding of the time intervals between events.

This method is designed because existing temporal methods, whether fixed or learned, fail to learn event cycles based on event features, despite the recognized importance of incorporating semantic features for accurately representing periodic patterns in real-world data.


In [28]:
print(rag_v1("what is significance of event prediction", rag_retriever, model, top_k=3).content)

<class 'langchain_ollama.chat_models.ChatOllama'>
Retrieving documents for query: 'what is significance of event prediction'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts.


Batches: 100%|██████████| 1/1 [00:00<00:00, 85.28it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents above the similarity threshold.


Based on the context provided, the significance of event prediction relates to the ability to model the influence of historical events on future outcomes.

Specifically:

1.  **Modeling Historical Impact:** The Weibull distribution is emphasized because it offers a flexible approach to modeling "how past events impact future probabilities" through its variable hazard function (which can be increasing, decreasing, or constant).
2.  **Quantitative Measurement:** The prediction is quantified using the **Negative Log-Likelihood (NLL) of event time** ($L_t$) as the loss function for event time prediction.
3.  **Comparative Goal:** The overall significance is evaluated through comparative studies, aiming to compare proposed models (like XTSFormer) with baseline models in neural networks.


In [29]:
print(rag_v2("what is significance of event prediction", rag_retriever, model, top_k=3).content)

<class 'langchain_ollama.chat_models.ChatOllama'>
Retrieving documents for query: 'what is significance of event prediction'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts.


Batches: 100%|██████████| 1/1 [00:00<00:00, 84.26it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents above the similarity threshold.


Based on the provided documents, the significance of event prediction is linked to modeling complex, real-world scenarios, specifically within **"Irregular-Time Event Prediction in Clinical Applications."**

The documents emphasize the technical importance of accurate prediction by highlighting:

1.  **Modeling Historical Influence:** The primary significance is the ability to model how **past events impact future probabilities**.
2.  **Choosing the Right Model:** Prediction is significantly advanced by using distributions like the **Weibull distribution**, which offers a flexible variable hazard function (increasing, decreasing, or constant). This approach is crucial because it accounts for history, unlike models that assume a constant intensity and uniform likelihood regardless of past occurrences.
3.  **Developing Advanced Methods:** The field requires advanced comparative evaluation, such as the proposed XTSFormer, to accurately forecast these temporal events.
